In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("creditcard.csv")
df['Amount'] = df['Amount'] * 111

df.head()
df.info()
df['Class'].value_counts(normalize=True) * 100

In [ ]:
# Convert Time to Hour of Day
df['Hour'] = (df['Time'] // 3600) % 24

In [ ]:
# Separate fraud and non-fraud
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

print(f"Total transactions: {len(df)}")
print(f"Fraud: {len(fraud)} ({len(fraud)/len(df)*100:.3f}%)")
print(f"legit: {len(legit)} ({len(legit)/len(df)*100:.3f}%)")

In [ ]:
# Hourly patterns.
plt.figure(figsize=(12, 6))
sns.histplot(legit['Hour'], bins=24, kde=False, color='blue', label='legit', stat='density')
sns.histplot(fraud['Hour'], bins=24, kde=False, color='red', label='Fraud', stat='density')
plt.title('Normalized Time-of-Day Distribution: Fraud vs. legit')
plt.xlabel('Hour of Day')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
hourly_fraud = df.groupby('Hour')['Class'].mean()

plt.figure()
hourly_fraud.plot()
plt.title("Fraud Rate by Hour")
plt.show()

In [ ]:
# Frequency per hour
freq_legit = legit['Hour'].value_counts(normalize=True).sort_index()
freq_fraud = fraud['Hour'].value_counts(normalize=True).sort_index()

freq_df = pd.DataFrame({'legit': freq_legit, 'Fraud': freq_fraud}).fillna(0)
freq_df.plot(kind='bar', figsize=(12, 6))
plt.title('Normalized Frequency of Transactions by Hour')
plt.xlabel('Hour')
plt.ylabel('Normalized Frequency')
plt.show()

# Overall transaction frequency stats
print("Average transactions per hour (legit):", len(legit) / 48)  # ~2 days = 48 hours
print("Average transactions per hour (fraud):", len(fraud) / 48)

Findings: Non-fraud frequency aligns with daily cycles (higher during waking hours). Fraud frequency is lower overall but proportionally higher at night, indicating opportunistic patterns. Average hourly non-fraud: ~5,923; fraud: ~10.

## Statistical Comparison

In [ ]:
# Compare Mean Transaction Amount (T-Test)

from scipy.stats import ttest_ind

fraud_amt = fraud['Amount'].mean()
legit_amt = legit['Amount'].mean()
print(f"Mean amount (legit): ₹{legit_amt:.2f}")
print(f"Mean amount (fraud): ₹{fraud_amt:.2f}")

# T-test for significance
t_stat, p_val = ttest_ind(legit['Amount'], fraud['Amount'], equal_var=False)
print(f"T-stat: {t_stat:.2f}, P-value: {p_val:.2e} (significant if <0.05)")

Findings: Legit mean amount: ~₹88.29. Fraud mean amount: ~₹122.21. The difference is statistically significant (p-value << 0.05), confirming fraud amounts are higher on average.

In [ ]:
# KS test for Amount distribution shift
from scipy.stats import ks_2samp

ks_stat, ks_p = ks_2samp(legit['Amount'], fraud['Amount'])
print(f"KS stat: {ks_stat:.2f}, P-value: {ks_p:.2e} (high stat indicates shift)")

# Plot distributions
plt.figure(figsize=(12, 6))
sns.kdeplot(legit['Amount'], color='blue', label='Legit')
sns.kdeplot(fraud['Amount'], color='red', label='Fraud')
plt.title('Amount Distribution: Fraud vs. Legit')
plt.xlabel('Amount')
plt.xlim(0, 111000)  # Zoom in to avoid long tails
plt.legend()
plt.show()

Findings: KS stat ~0.43 (p-value ~0), indicating a significant shift—fraud distributions are skewed toward higher values with more variability. Legit clusters around low amounts (<₹100), while fraud spreads wider, often >₹100.

## Risk Segmentation

In [ ]:
# Create Amount Buckets
bins = [0, 5550, 11100, 55500, 111000, 555000]
labels = ['0-5550', '5550-11100', '11100-55500', '55500-111000', '111000+']
df['Amount_Bucket'] = pd.cut(df['Amount'], bins=bins, labels=labels)

# Fraud Rate per Bucket
bucket_risk = df.groupby('Amount_Bucket')['Class'].mean() * 100
bucket_risk

In [ ]:
bucket_risk.plot(kind='bar', figsize=(8, 5))
plt.title('Fraud Rate by Amount Bucket (%)')
plt.ylabel('Fraud Rate')
plt.show()

Typical Findings
- 0–5550 → Very low fraud rate
- 11100–55500 → Elevated risk
- 55500+ → Significantly higher fraud %
- This reveals risky transaction ranges.

In [ ]:
# Rule-Based Flags
df['High_Risk_Flag'] = np.where(
    (df['Amount'] > 55500) & (df['Hour'].isin([0,1,2,3,4])), 1, 0
)
pd.crosstab(df['High_Risk_Flag'], df['Class'])

# Financial Impact Analysis

In [ ]:
# Total fraud loss
total_fraud_loss = df[df['Class'] == 1]['Amount'].sum()
total_fraud_loss

In [ ]:
# Fraud % of total transactions
fraud_percentage = df['Class'].mean() * 100
fraud_percentage

In [ ]:
fraud_value_percentage = (
    total_fraud_loss / df['Amount'].sum()
) * 100

fraud_value_percentage

Findings: Total fraud loss: 60,124.58 (from dataset calculations). This is 0.27% of total transaction volume (~₹25M), higher than the transaction fraud rate due to elevated amounts.
Even though fraud is rare in frequency, its financial weight is disproportionately high.

### Fraud Heatmap (Hour vs Amount Bucket)

In [ ]:
heatmap_data = df.pivot_table(
    values='Class',
    index='Hour',
    columns='Amount_Bucket',
    aggfunc='mean'
)

plt.figure()
sns.heatmap(heatmap_data, cmap='Reds')
plt.show()

Reveals:
- Concentrated fraud zones
- Night + High amount = hotspot

In [ ]:
# Time Trend
df.groupby('Hour')['Class'].mean().plot()
plt.show()

### Risk Segmentation Summary Table

| Amount Bucket | Fraud Rate | Risk Level |
| ------------- | ---------- | ---------- |
| 0-5550        | Very Low   | Low        |
| 5550-11100    | Low        | Medium     |
| 11100-55500   | Moderate   | High       |
| 55500+        | High       | Very High  |


## Summary

### Strategic Recommendations

- Implement rule: Flag transactions > ₹55500 during midnight hours
- Increase authentication friction for high-risk buckets
- Real-time alerting during fraud spike hours
- Dynamic transaction scoring using amount + time